# Reproducing Bell Vila et al. (2026) from dandiset 001678

Paper. Bell Vila M, Koletar M, Dorr A, Trevisiol A, Goubran M, Sled JG, Bazzigaluppi P,
Stefanovic B. Parametrizing alternating current stimulation for neuromodulation. *Imaging
Neuroscience* 4 (2026). [doi:10.1162/imag.a.1066](https://doi.org/10.1162/imag.a.1066). The claims
file (`claims/001678/10.1162-imag.a.1066.yaml`) was written from the version of record, open
access in Europe PMC (PMC12779756), with its supplementary material; this attempt read the same
text.

Dandiset. [DANDI:001678](https://dandiarchive.org/dandiset/001678/0.260227.1824), version
0.260227.1824 (doi:10.48324/dandi.001678/0.260227.1824). The paper cites no DANDI version: its data
statement points to Science Data Bank (doi:10.57760/sciencedb.32375), whose record answers
"PERMISSION NO ACCESS" without a login. 0.260227.1824 is the only published version, so it is
pinned; the draft holds the same 22 files (checked below).

What the dandiset holds (shown in the data check). The 9 files of the 1-minute paradigm hold
one channel per trial, and that channel is the stimulation command (a sinusoid at the trial's
frequency whose peak is the current in uA, zero outside stimulation). They hold no recording
channel: no multi-unit activity, no LFP and no stimulation monitor. The 13 files of the 20-minute
paradigm hold a 16-channel stream at 12207 Hz that carries both the LFP and spikes (the
multi-unit band is not filtered out), but no sorted units. The sham recording is not in the
dandiset. Every claim on the 1-minute recordings or the sham recording is blocked for missing
data. The claims on 20-minute spikes need the recordings sorted: by the estimate made below,
sorting the 13 rats exceeds the notebook's 60-minute limit, and these claims, with the LFP
claims that select channels by their neurons, are blocked as too large. The claim tested is the electric field of the 20-minute experiments, from the stimulation
artifact.

| Claim | Tested here |
|---|---|
| efield-20min-median | tested |
| efield-1min-medians | blocked, data missing (no recording channel in the 1-minute files) |
| fem-depth-falloff, fem-lognormal-fits, fem-ellipsoid-axes | blocked, data missing (depend on efield-1min-medians; the model and solver are not public) |
| units-1min and the 25 claims on 1-minute spikes or LFP (baseline-fr, baseline-ppc, table1-any-change, fr-change-more-common-than-timing, fr-change-mostly-negative, ppc-change-mostly-increase, table2-magnitudes, fr-change-within-2hz, fig4-distributions, the four lmer claims, fig6a, fig6b, fig6d, fig7, fig8a, fig8b, fig8c, fig9a, fig9b, fig9cd, fig10a, lfp-1min-no-change) | blocked, data missing |
| units-20min, 20min-during-percentages, 20min-post-percentages, 20min-magnitudes | blocked, too large (spike sorting of 13 rats, estimate below) |
| the four power and four PAC claims on channels near affected neurons | blocked, too large (the channels are chosen by the neurons of 20min-during-percentages) |
| sham-no-fr-change, sham-lfp-baseline | blocked, data missing (no sham recording) |

In [ ]:
import time
from concurrent.futures import ProcessPoolExecutor

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests

import acs

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 300)
np.set_printoptions(precision=3, suppress=True, linewidth=160)

started = time.time()
assets = acs.assets()
print(f"dandiset {acs.DANDISET_ID} version {acs.DANDISET_VERSION}: {len(assets)} files, "
      f"{sum(a['size'] for a in assets) / 1e9:.1f} GB")
info = acs.version_info()
print("name:", info["metadata"]["name"])
print("description:", info["metadata"]["description"])

## Data check

Each check of the metadata against the values is stated under its output.

In [ ]:
versions = requests.get(f"{acs.API}/dandisets/{acs.DANDISET_ID}/versions/", timeout=60).json()["results"]
print("versions:", [(v["version"], v["status"], v["asset_count"], v["size"]) for v in versions])
draft = requests.get(f"{acs.API}/dandisets/{acs.DANDISET_ID}/versions/draft/assets/?page_size=200", timeout=60).json()["results"]
pub = {(a["path"], a["size"]) for a in assets}
dr = {(a["path"], a["size"]) for a in draft}
print("draft files equal to the published version's (path and size):", pub == dr)

with ProcessPoolExecutor(max_workers=4) as ex:
    summaries = list(ex.map(acs.file_summary, assets))
rows = []
for s in summaries:
    ser = s["series"]
    rows.append({
        "subject": s["subject_id"], "paradigm": s["paradigm"], "sex": s["sex"], "age": s["age"],
        "species": s["species"], "GB": round(s["size_GB"], 2), "n_series": len(ser),
        "channels": ser[0]["shape"][1], "channel_names": ",".join(sorted(set(s["channel_names"])))[:40],
        "rate_hz": sorted({x["rate"] for x in ser}), "conversion": sorted({x["conversion"] for x in ser}),
        "unit": sorted({x["unit"] for x in ser}), "minutes_total": round(sum(x["shape"][0] / x["rate"] for x in ser) / 60, 1),
        "units_table": s["has_units"], "intervals": s["has_intervals"],
        "other_groups": {k: v for k, v in s["groups"].items() if v},
        "electrode_columns": ",".join(s["electrode_columns"]), "locations": ",".join(s["locations"]),
        "session_description": s["session_description"],
    })
files = pd.DataFrame(rows)
print(files.drop(columns=["channel_names"]).to_string(index=False))
print()
print(files.groupby(["paradigm", "sex"]).size().rename("rats").to_string())
print("\nChannel names, 1-min files:", sorted({n for s in summaries if s["paradigm"] == "1-min" for n in s["channel_names"]}))
print("Channel names, 20-min files:", sorted({n for s in summaries if s["paradigm"] == "20-min" for n in s["channel_names"]},
                                             key=lambda n: int(n.split()[-1])))
print("Any units table, intervals, processing, analysis or stimulus data:",
      bool(files.units_table.any() or files.intervals.any() or any(files.other_groups.map(bool))))

The paper (Methods 2.1, 2.4) has 9 rats in the 1-minute paradigm (4 male, 5 female) and 13 in the
20-minute paradigm (7 male, 6 female). The files have 9 and 13 subjects, 4 male and 5 female in the
1-minute set, as the paper says, but 6 male and 7 female in the 20-minute set, the reverse of the
paper's count. Every file has an electrodes table with only `location` ("unknown"), `group`,
`group_name` and `channel_name`: no positions, so the 4 x 4 layout of the array is not in the
files. No file has a units table, trial or epoch intervals, or a processing module. The series are
in "volts" with a conversion of 1e-6, which the next cells test.

In [ ]:
one_min = [a for a in assets if acs.paradigm(a["path"]) == "1-min"]
with ProcessPoolExecutor(max_workers=4) as ex:
    trials = pd.DataFrame([r for rows in ex.map(acs.stim_trials, one_min) for r in rows])
trials["freq_hz"] = trials.freq_fft_hz.round(0)
trials["amp"] = trials.peak.round(0)
print("Trials (one-channel series) per rat:", trials.groupby("subject").size().to_dict())
print("Total trials:", len(trials), "; paper, Methods 2.4: 20 per rat, 180 for 9 rats")
print("\nWaveform frequency from the spectrum (Hz), values:", sorted(trials.freq_hz.unique()))
print("Zero-crossing frequency minus spectral frequency, max abs (Hz):",
      round((trials.freq_zero_cross_hz - trials.freq_fft_hz).abs().max(), 3))
print("Peak values:", sorted(trials.peak.round(2).unique()))
print("Peak / RMS, range:", (trials.peak / trials.rms).min().round(4), "to", (trials.peak / trials.rms).max().round(4),
      "(a sinusoid gives 1.4142)")
print("Nonzero samples outside the waveform, total:", int(trials.nonzero_outside.sum()))
print("\nOnset within the series (s): %.2f to %.2f; duration (s): %.2f to %.2f; series length (s): %.1f to %.1f" % (
    trials.onset_s.min(), trials.onset_s.max(), trials.duration_s.min(), trials.duration_s.max(),
    trials.length_s.min(), trials.length_s.max()))
print("\nTrials by frequency (rows) and peak (columns), all rats:")
print(pd.crosstab(trials.freq_hz, trials.amp).to_string())
design = trials[trials.freq_hz.isin([5, 10, 20, 40]) & trials.amp.isin([50, 100, 200, 400])]
per_rat = design.groupby("subject").apply(lambda g: pd.Series({
    "trials_5_to_40Hz": len(g), "distinct_combinations": g.groupby(["freq_hz", "amp"]).ngroups,
    "repeated": int((g.groupby(["freq_hz", "amp"]).size() > 1).sum())}), include_groups=False)
per_rat["trials_140Hz"] = trials[trials.freq_hz == 140].groupby("subject").size()
per_rat["other"] = trials[~trials.index.isin(design.index) & (trials.freq_hz != 140)].groupby("subject").size()
print("\nPer rat, of the 16 combinations of 5 to 40 Hz and 50 to 400 uA:")
print(per_rat.fillna(0).astype(int).to_string())
missing = {s: sorted(set((f, a) for f in [5, 10, 20, 40] for a in [50, 100, 200, 400]) -
                     set(map(tuple, g[["freq_hz", "amp"]].values))) for s, g in design.groupby("subject")}
print("Combinations without a trial:", {k: v for k, v in missing.items() if v})

Each one-channel series of a 1-minute file is the stimulation command, not a recording: it is
exactly zero for the first 60.2 s and the last minute, and during the minute between is a pure
sinusoid (peak over RMS of 1.414) at 5, 10, 20, 40 or 140 Hz whose peak is 50, 100, 200 or 400 to
within 0.2%, the currents of Methods 2.4 in uA. The channel name "S1r 1" matches the stream
`uS1r` that the paper's MUAartrem.m reads as the "stimulation channel"; the multi-unit stream
(`Wav2`) and the monitor (`MonA`) that the code uses are not stored. The stated unit (volts, times
1e-6) is wrong for this channel: the values are the current in uA. The trial timing agrees with
the paradigm (1 min of baseline, 1 min of stimulation, then the post-stimulus minute). The files
give the frequency and amplitude of every trial, which the paper's analysis needs, but no neural
data to apply them to. The counts of trials differ from the paper's 20 per rat (printed above), and
one trial at 100 Hz and 20 uA is outside the paper's design.

In [ ]:
twenty = [a for a in assets if acs.paradigm(a["path"]) == "20-min"]
with ProcessPoolExecutor(max_workers=4) as ex:
    profiles = list(ex.map(acs.band_power_profile, twenty))
prof = pd.DataFrame([{"subject": p["path"].split("/")[0], **{k: v for k, v in p["bands_psd"].items()},
                      "median_abs": p["raw_median_abs"], "p99_abs": p["raw_p99_abs"],
                      "500-3000Hz_std": p["mua_band_std"], "crossings_6MAD_per_s": p["mua_crossings_per_s_median"],
                      "kurtosis_500-3000Hz": p["mua_kurtosis_median"]} for p in profiles])
print("Welch power spectral density of the stored values (units^2/Hz), 30 s of baseline from minute 5, mean over 16 channels:")
print(prof.set_index("subject").iloc[:, :9].map(lambda v: f"{v:.1e}").to_string())
ratio = prof["100-200 Hz"] / prof["500-1000 Hz"]
print("\nPSD 100-200 Hz over 500-1000 Hz, range: %.0f to %.0f" % (ratio.min(), ratio.max()))
print("PSD 500-1000 Hz over 3000-6000 Hz, range: %.1f to %.1f" % ((prof["500-1000 Hz"] / prof["3000-6000 Hz"]).min(),
                                                                (prof["500-1000 Hz"] / prof["3000-6000 Hz"]).max()))
print("\nStored values, median |x| and 99th percentile |x| over channels:")
print(prof[["subject", "median_abs", "p99_abs", "500-3000Hz_std", "crossings_6MAD_per_s", "kurtosis_500-3000Hz"]].to_string(index=False))
print("\nIf the stored values were scaled by the conversion 1e-6 to volts, the median |LFP| would be "
      f"{prof.median_abs.median() * 1e-6 * 1e9:.2f} nV; read as volts it is {prof.median_abs.median() * 1e3:.2f} mV.")

In [ ]:
spikes = pd.DataFrame([{"subject": p["path"].split("/")[0], "line_60Hz_ratio": p["line60_ratio"],
                        "events_per_s_busiest_channel": p["events_per_s_max"], "busiest_channel": p["busiest_channel"],
                        "share_coincident_on_other_channel": p["coincident_share"],
                        "mean_event_trough_uV": p["trough_uV"], "trough_width_ms": p["trough_width_ms"]} for p in profiles])
print("60 Hz line over its neighbours (50 to 70 Hz), negative 500-3000 Hz peaks beyond 6 MAD on the busiest channel,")
print("and the mean unfiltered waveform at those peaks (trough and its width at half depth):")
print(spikes.round(3).to_string(index=False))

The 20-minute files hold one 16-channel stream at 12207 Hz. Methods 2.3 describe two: the LFP
(0.3 to 300 Hz, notch-filtered at 60, 120 and 180 Hz, stored at 12 kHz) and the multi-unit band
(500 to 3000 Hz, stored at 24 kHz). This stream has the LFP's rate but is neither low-passed at
300 Hz nor notch-filtered: the 60 Hz line stands 10 to several hundred times above its
neighbours, and the spectral density above 300 Hz (printed above) falls slowly, in some rats by
less than tenfold from 100-200 Hz to 500-1000 Hz. In several rats the 500 to 3000 Hz band carries
spikes: negative peaks beyond 6 MAD at up to tens per second on the busiest channel, in most rats
seldom coincident with other channels 0.5 mm away (in sub-2023-03-22 they are coincident, and are
more likely noise), with an unfiltered mean waveform whose trough is 50 to 180 uV deep and a few
tenths of a millisecond wide. So the files hold what spike sorting needs for
the 20-minute experiments, at half the paper's sampling rate. In other rats few such events
appear in the 30 s examined. The stated conversion of 1e-6 would make the signals tenths of a
nanovolt, far below the noise of any amplifier; read as volts, the median of about 0.1 mV and the
99th percentiles below 7 mV are typical of cortical recordings, and the spike troughs of about
100 uV are typical of extracellular spikes. The values are taken as volts, and the conversion
factor is recorded as a metadata error.

In [ ]:
with ProcessPoolExecutor(max_workers=4) as ex:
    prints = pd.DataFrame([r for rows in ex.map(acs.series_fingerprints, assets) for r in rows])
nonzero = prints[~prints.all_zero]
dup = nonzero[nonzero.duplicated("sha1_first_second", keep=False)]
print("Series:", len(prints), "; with a nonzero first second:", len(nonzero), "; all-zero first second:", int(prints.all_zero.sum()))
print("Series whose first second is identical to another's:", len(dup))
print(dup.to_string(index=False) if len(dup) else "")
lengths = trials.groupby("subject").apply(lambda g: tuple(zip(g.freq_hz, g.amp)), include_groups=False)
print("1-min rats with the same sequence of trial parameters as another rat:",
      int(lengths.duplicated(keep=False).sum()))

No 20-minute series repeats another (the first second of each differs), and no two 1-minute rats
share the same sequence of trial parameters, so no recording was found deposited twice. The
1-minute series all start with zeros (the command before stimulation), so their first seconds
cannot be compared; their parameter sequences can.

In [ ]:
t0 = time.time()
with ProcessPoolExecutor(max_workers=4) as ex:
    peaks = list(ex.map(acs.cycle_peaks, twenty))
print(f"cycle peaks of 13 rats in {time.time() - t0:.0f} s")
timing = pd.DataFrame([{
    "subject": r["subject"], "series_min": r["series_s"] / 60, "baseline_min": r["onset_s"] / 60,
    "stimulation_min": (r["offset_s"] - r["onset_s"]) / 60, "after_offset_min": (r["series_s"] - r["offset_s"]) / 60,
    "cycles": r["n_cycles"], "cycles_per_s": r["n_cycles"] / (r["offset_s"] - r["onset_s"] - 2 * acs.EDGE_S)} for r in peaks])
print(timing.round(2).to_string(index=False))
later = files[files.paradigm == "20-min"][["subject", "n_series", "minutes_total"]]
print("\nSeries per 20-min file and total minutes:", dict(zip(later.subject, zip(later.n_series, later.minutes_total))))

The files have no epochs, so the stimulation is found from its 40 Hz artifact (1-s resolution) in
the first series of each file. In every rat the artifact lasts 20 minutes (printed above), as
Methods 2.4 state, at 40 cycles per second. The baseline before it is about 20 minutes in most
rats and up to 31 minutes in one. The first series continues 5.6 to 20.9 minutes after offset,
less than the 10 minutes that the paper compares after offset in four rats (the paper's
nulldistribution.py shortens the window when the first post-stimulation segment is shorter); the
further series are the later post-stimulation recordings that the dandiset's description names.
There is no trial table whose clock the streams could be compared with; the stimulation times come
from the data, and the analysis below uses only the stimulation period.

## Spike sorting of the 20-minute recordings: how long it would take

units-20min and every claim built on its neurons need the 20-minute recordings sorted, as the
paper did with SpyKING CIRCUS (Methods 2.6) over each rat's whole recording (the paper's
concatdataforspikesorting.py). The sorter is timed here on 5 and 15 minutes of one rat (from
minute 5 of its first series), and the time is extrapolated, as a line through the two, to the
first series of every rat, which holds the baseline, the stimulation and the 10 minutes after it.

In [ ]:
rat = next(a for a in twenty if a["path"].startswith("sub-2023-04-28"))
runs = [acs.sorting_timing(rat, m) for m in (5, 15)]
for r in runs:
    print(f"{r['subject']}: {r['minutes']:.0f} min sorted in {r['seconds']:.0f} s, {r['units']} units; "
          f"spikes per unit from {r['spikes_per_unit'][0]} to {r['spikes_per_unit'][-1]}")
slope = (runs[1]["seconds"] - runs[0]["seconds"]) / (runs[1]["minutes"] - runs[0]["minutes"])
intercept = runs[0]["seconds"] - slope * runs[0]["minutes"]
est = intercept + slope * timing.series_min
print(f"\nseconds = {intercept:.0f} + {slope:.1f} x minutes")
print("Estimated sorting time per rat (min):", dict(zip(timing.subject, (est / 60).round(1))))
print(f"Estimated sorting time for the 13 rats: {est.sum() / 60:.0f} min, against the notebook limit of 60 min; "
      f"this notebook had run {(time.time() - started) / 60:.0f} min before this estimate.")

The sorter finds units in a few minutes of one rat, which confirms that the 20-minute files hold
sortable spikes. Sorting the 13 first series, by the estimate printed above, takes longer than a
notebook may run (60 minutes), before the permutation tests, the phase extraction and the LFP
analyses that build on the neurons. units-20min is therefore blocked as too large, and with it
the claims that use its neurons: 20min-during-percentages, 20min-post-percentages,
20min-magnitudes, and the four power and four PAC claims, whose channels are those near the
affected neurons. The sorting was not run on the whole recordings.

## Claim efield-20min-median

> The median electric field in the 20-minute 100 uA experiments was 1.65 mV/mm.
> (Results 3.4, paragraph 1; Figure 1D. Expected 1.65 +/- 0.21 mV/mm.)

Criterion: as efield-1min-medians (Methods 2.8, 40 Hz band), over the 13 rats, with the same
readings: (a) the median over rats of each rat's average field, (b) the median over all
channel-pair estimates of all rats; the field per axis read as (i) each neighbour pair, (ii) the
largest pair per axis and rat; a zero-phase Butterworth band of +/- 1 Hz on the LFP stream.
Reading (i) is run two ways, since the paper does not say how a pair along one axis is matched
with a pair along the other for the vector addition: per 2 x 2 cell (the mean of its two
differences along each axis) and per electrode (the differences to its neighbours to the right and
below). Methods 2.8 apply Grubbs' test to "electric field values" without saying over which set;
it is run over all estimates of all rats together (main), within each rat, and not at all.

In [ ]:
rng = np.random.default_rng(1678)  # this section's generator
pk = np.array([r["peak_mean"] for r in peaks])  # rats x 16, volts as stored
subjects = [r["subject"] for r in peaks]
print("Average cycle peak per channel (mV), channels aw_1 to aw_16 laid out row by row:")
for s, v in zip(subjects, pk):
    print(s)
    print(acs.grid(v) * 1e3)
print("\nAll average peaks, range (mV): %.3f to %.3f" % (pk.min() * 1e3, pk.max() * 1e3))
cv = np.array([r["peak_cv"] for r in peaks])
print("Spread of cycle peaks within a channel (CV), median %.3f, range %.3f to %.3f" % (np.median(cv), cv.min(), cv.max()))
half = np.array([r["peak_first_half"] / r["peak_second_half"] for r in peaks])
print("First-half over second-half average peak, per channel, median %.2f, range %.2f to %.2f" % (np.median(half), half.min(), half.max()))

In [ ]:
# The channel layout is not in the files or the paper. Test the assumed order (aw_1 to aw_16 row by
# row) against the serpentine order and random orders: under the right layout the artifact, which
# falls off with distance from the stimulating pads, should vary smoothly across the grid.
n_perm = 2000
rows = []
for s, v in zip(subjects, pk):
    r_row = acs.roughness(acs.grid(v))
    r_serp = acs.roughness(acs.grid(v, acs.SERPENTINE))
    r_rand = np.array([acs.roughness(acs.grid(v, rng.permutation(16))) for _ in range(n_perm)])
    rows.append({"subject": s, "rough_row_by_row": r_row * 1e6, "rough_serpentine": r_serp * 1e6,
                 "rough_random_median": np.median(r_rand) * 1e6,
                 "frac_random_smoother_than_row": np.mean(r_rand <= r_row),
                 "frac_random_smoother_than_serp": np.mean(r_rand <= r_serp)})
maps = pd.DataFrame(rows)
print("Sum of squared neighbour differences (mV^2) under each order, and the share of", n_perm, "random orders as smooth or smoother:")
print(maps.round(4).to_string(index=False))
print("\nRats where row-by-row is smoother than serpentine:", int((maps.rough_row_by_row < maps.rough_serpentine).sum()), "of", len(maps))

In [ ]:
def estimates(order=None):
    out = []
    for s, v in zip(subjects, pk):
        g = acs.grid(v, order)
        F = acs.fields(g)
        node = np.hypot(F["ex"][:3, :3], F["ey"][:3, :3]).ravel()
        out.append({"subject": s, "cell": F["cells"], "node": node, "maximal": np.array([F["maximal"]])})
    return out


def summarise(est, kind, grubbs):
    pooled = np.concatenate([e[kind] for e in est])
    if grubbs == "pooled":
        kept, removed = acs.grubbs(pooled)
        keep_sets = []
        for e in est:
            keep_sets.append(np.array([x for x in e[kind] if not np.any(np.isclose(x, removed))]))
    elif grubbs == "per rat":
        keep_sets = [acs.grubbs(e[kind])[0] if len(e[kind]) > 2 else e[kind] for e in est]
        removed = np.array([0] * (len(pooled) - sum(len(k) for k in keep_sets)))
    else:
        keep_sets = [e[kind] for e in est]
        removed = np.array([])
    allkept = np.concatenate(keep_sets)
    per_rat = np.array([k.mean() for k in keep_sets if len(k)])
    return {"estimates": len(pooled), "removed": len(removed),
            "(a) median of rat means": np.median(per_rat), "(b) median of all estimates": np.median(allkept),
            "rat means min": per_rat.min(), "rat means max": per_rat.max(),
            "IQR of all (b)": np.subtract(*np.percentile(allkept, [75, 25])), "SD of rat means": per_rat.std(ddof=1)}


rows = []
for order_name, order in [("row by row (assumed)", None), ("serpentine", acs.SERPENTINE)]:
    est = estimates(order)
    for kind, label in [("cell", "(i) per 2x2 cell"), ("node", "(i) per electrode"), ("maximal", "(ii) largest pair per axis")]:
        for gr in ["pooled", "per rat", "none"]:
            if kind == "maximal" and gr == "per rat":
                continue
            rows.append({"layout": order_name, "pairs": label, "Grubbs": gr, **summarise(est, kind, gr)})
table = pd.DataFrame(rows)
lo, hi = 1.65 - 0.21, 1.65 + 0.21
table["(a) within 1.44-1.86"] = table["(a) median of rat means"].between(lo, hi)
table["(b) within 1.44-1.86"] = table["(b) median of all estimates"].between(lo, hi)
print("Electric field (mV/mm), 13 rats, expected 1.65 +/- 0.21:")
print(table.round(3).to_string(index=False))

In [ ]:
est = estimates()
per_rat = pd.DataFrame([{"subject": e["subject"], "cell mean": e["cell"].mean(), "cell median": np.median(e["cell"]),
                         "node mean": e["node"].mean(), "maximal": e["maximal"][0],
                         "max |Ex| pair": np.abs(acs.fields(acs.grid(v))["ex"]).max(),
                         "max |Ey| pair": np.abs(acs.fields(acs.grid(v))["ey"]).max()}
                        for e, v in zip(est, pk)])
print("Per rat, assumed layout, no outlier removal (mV/mm):")
print(per_rat.round(3).to_string(index=False))
allcells = np.concatenate([e["cell"] for e in est])
print("\nAll 117 cell estimates: range %.3f to %.3f mV/mm; median %.3f" % (allcells.min(), allcells.max(), np.median(allcells)))

fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].hist(allcells, bins=30, color="0.4")
ax[0].axvline(1.65, color="C3", label="paper 1.65")
ax[0].axvspan(lo, hi, color="C3", alpha=0.15)
ax[0].set_xlabel("field per 2x2 cell (mV/mm)"); ax[0].set_ylabel("estimates"); ax[0].legend()
ax[0].set_title("all 13 rats; no estimate outside the axes")
med = np.median(np.array([acs.grid(v) for v in pk]) * 1e3, axis=0)
im = ax[1].imshow(med, cmap="viridis"); plt.colorbar(im, ax=ax[1], label="median peak (mV)")
ax[1].set_title("median average peak over rats, row-by-row layout")
plt.tight_layout(); plt.show()
print("Median-over-rats peak grid range (mV): %.3f to %.3f" % (med.min(), med.max()))

In [ ]:
# Variants that cannot change the reading but show how sensitive the number is.
# 1) The band: +/- 2 Hz instead of +/- 1 Hz changes the filtered amplitude of a pure 40 Hz line by
#    less than the filter's ripple; computed on one rat to bound it.
r0 = twenty[0]
f = acs.open_asset(r0["asset_id"])
d = f["acquisition/ElectricalSeries0/data"]
fs = float(f["acquisition/ElectricalSeries0/starting_time"].attrs["rate"])
on = peaks[0]["onset_s"]
from scipy import signal
y = signal.decimate(d[int((on + 300) * fs):int((on + 360) * fs), :].astype(float), acs.LFP_DECIMATE, axis=0)
fs2 = fs / acs.LFP_DECIMATE
res = {}
for hw in [1.0, 2.0]:
    sos = signal.butter(acs.BUTTER_ORDER, [40 - hw, 40 + hw], btype="band", fs=fs2, output="sos")
    yb = signal.sosfiltfilt(sos, y, axis=0)
    per = fs2 / 40
    idx = (np.arange(int(len(yb) / per) + 1) * per).astype(int)
    res[hw] = np.maximum.reduceat(np.abs(yb), idx[:-1], axis=0)[:-1].mean(0)
print(f"{peaks[0]['subject']}, 60 s from 5 min after onset: peaks with +/-2 Hz over +/-1 Hz, per channel:",
      (res[2.0] / res[1.0]).round(3))
F1 = acs.fields(acs.grid(res[1.0])); F2 = acs.fields(acs.grid(res[2.0]))
print("cell-field median, +/-1 Hz %.3f, +/-2 Hz %.3f mV/mm" % (np.median(F1["cells"]), np.median(F2["cells"])))
# 2) The unit: with the files' conversion factor applied the fields are a million times smaller.
print("Main reading with the stored conversion (1e-6) applied: %.2e mV/mm" % (table.iloc[0]["(a) median of rat means"] * 1e-6))

**Result: reproduced, resting on an assumed channel layout.** With channels aw_1 to aw_16 laid
out row by row and reading (i) taken per 2 x 2 cell, the median field is 1.834 mV/mm as the median
of the 13 rat means (a) and 1.525 mV/mm as the median of all 117 estimates (b), both within
1.65 +/- 0.21. Taken per electrode, reading (b) gives 1.648 (within) and reading (a) 1.890 (just
outside). Reading (ii), the largest pair per axis, gives 3.698, outside. Grubbs' test over the
pooled estimates removes none. The serpentine layout gives the same outcome (1.848 and 1.589 per
cell). The row-by-row layout is smoother than the serpentine one in 10 of 13 rats, and at most
1.35% of random layouts are as smooth in any rat, which supports a layout in rows of four but
does not prove this one. The spread between rats is wider than the paper's +/- 0.21: rat means
range from 0.810 to 3.052 mV/mm (SD 0.730). Widening the band to +/- 2 Hz changes the peaks by
under 1% in the rat checked.

## Summary

| Claim | Outcome |
|---|---|
| efield-20min-median | reproduced (assumed channel layout): 1.83 mV/mm (a), 1.53 (b), expected 1.65 +/- 0.21 |
| efield-1min-medians | blocked, data missing |
| fem-depth-falloff, fem-lognormal-fits, fem-ellipsoid-axes | blocked, data missing |
| units-1min and 25 claims on the 1-minute spikes or LFP | blocked, data missing |
| units-20min, 20min-during-percentages, 20min-post-percentages, 20min-magnitudes | blocked, too large |
| power-entrained-channels, power-reduced-entrainment-channels, power-excited-channels, power-suppressed-channels | blocked, too large |
| pac-entrained-channels, pac-reduced-entrainment-channels, pac-excited-channels, pac-suppressed-channels | blocked, too large |
| sham-no-fr-change, sham-lfp-baseline | blocked, data missing |

How the analysis differs from the paper's. The electric field is computed from the stored
16-channel stream of the 20-minute files, decimated to 1.2 kHz, with a +/- 1 Hz Butterworth band
at 40 Hz (the paper names no filter). The 4 x 4 layout of the channels is assumed (row by row),
as neither the files nor the paper give it. The stimulation period is found from the artifact, as
the files have no epochs. How the two orthogonal differences are paired for the vector addition is
not stated; two pairings are shown.

What was found about the dataset.
- The 1-minute files hold only the stimulation command channel ("S1r 1", the stream `uS1r`), one
  series per trial: the multi-unit recordings, the LFP and the stimulation monitor that the paper
  analysed are not there. The command gives each trial's frequency and amplitude.
- The command channel is labelled volts with a conversion of 1e-6, but its values are the current
  in uA; the 20-minute streams are labelled the same way, but their values are volts.
- The 20-minute files hold one 16-channel stream at 12207 Hz that is neither low-passed at 300 Hz
  nor notch-filtered, and carries spikes. No sorted units, trial or epoch tables, stimulation
  times or stimulation waveform are stored.
- The electrodes table has no positions, so the 4 x 4 layout of the array is not documented.
- The 1-minute rats have 15 to 23 trials (the paper says 20), three rats lack some of the 16
  combinations, and one trial at 100 Hz and 20 uA is outside the paper's design.
- The 20-minute subjects are 6 male and 7 female; the paper says 7 male and 6 female.
- The sham recording is not in the dandiset.